# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv("clientes_ecommerce.csv")
# TODO: mostrar 10 primeras filas
df.head(10)
# TODO: consultar shape, dtypes, info y número de duplicados
print(df.shape)
print(df.dtypes)
print(df.info())
print("Filas duplicadas: ", df.duplicated().sum())

(400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                400 non

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- **Muestras y columnas:** Hay 400 muestras (filas) y 12 columnas.
- **Target:** `compra_proximo_mes`.
- **Identificador de cliente:** `cliente_id`.
- **Filas duplicadas:** No, hay 0 filas duplicadas.

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [47]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean().mul(100)
# TODO: filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()

display(pd.DataFrame({"ausentes": ausentes, "porcentaje": porcentaje.round(2)}))
print("Filas con al menos un ausente:", filas_incompletas)

,ausentes,porcentaje
cliente_id,0,0.00
edad,0,0.00
antiguedad_meses,5,1.25
ingresos_anuales,4,1.00
gasto_mensual,3,0.75
num_compras,0,0.00
devoluciones,0,0.00
dispositivo,0,0.00
zona,0,0.00
nivel_fidelidad,0,0.00


Filas con al menos un ausente: 12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1,25% |
| `ingresos_anuales` | 4 | 1,00% |
| `gasto_mensual` | 3 | 0   ,75% |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No, no las eliminaría. El dataset solo tiene 400 muestras y el porcentaje de valores ausentes es muy bajo (entre el 0,75 % y el 1,25 % por columna), por lo que borrar filas supondría perder información valiosa innecesariamente cuando podemos imputar esos pocos datos (por ejemplo, usando la mediana) sin distorsionar el análisis.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [48]:
# TODO: mínimo y máximo de edad
# TODO: localizar registros con edad > 120
# Pista: df.loc[df["edad"] > 120, ["cliente_id", "edad"]]


print("Edad mínima:", df["edad"].min())
print("Edad máxima:", df["edad"].max())
###############

df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad mínima: 18
Edad máxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: La convertiría en ´NaN´ para imputarla después con la mediana (o eliminaría esa única fila). No se debe cambiar a 95 porque sería inventar un dato por simple suposición sin evidencia real, introduciendo información falsa.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | edad, antiguedad_meses, ingresos_anuales, gasto_mensual, num_compras, devoluciones, satisfaccion |
| Categóricas nominales | dispositivo, zona |
| Categóricas ordinales | nivel_fidelidad |
| Identificador | cliente_id |
| Target | compra_proximo_mes |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: 

`zona` es nominal porque sus categorías son etiquetas sin orden ni jerarquía entre sí.

`nivel_fidelidad` es ordinal porque sus categorías siguen una jerarquía natural de menor a mayor nivel (Bronce < Plata < Oro < Platino).

`No incluiría cliente_id` porque es un simple código identificador sin valor predictivo; incluirlo haría que el modelo memorice registros concretos en lugar de aprender patrones generales.

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [49]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
# TODO: y = ... (mapear No:0, Sí:1)
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
# TODO: X.shape, y.shape, y.value_counts()
print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

### Separamos las variables predictoras (`X`, eliminando el identificador y el target) 
# de la variable objetivo (`y`, codificada en binario como `0` y `1`). 


X: (400, 10)
y: (400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: Hay 10 características predictoras iniciales (las 10 columnas de X, tras quitar cliente_id y la variable objetivo compra_proximo_mes).

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [50]:
# TODO: completar
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=..., random_state=..., stratify=...
# )

X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42, stratify=y
)
# TODO: mostrar las dimensiones de los cuatro conjuntos


print("Entrenamiento :" ,X_train.shape, y_train.shape)
print("test  :" ,X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Entrenamiento : (320, 10) (320,)
test  : (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: 

Muestras: Hay 320 muestras en el conjunto de entrenamiento (80 %) y 80 en el de test (20 %).

stratify=y: (Sirve para mantener en ambos conjuntos exactamente la misma proporción de clases de la variable objetivo (~61,5 % de 0 y ~38,5 % de 1), evitando que por azar un conjunto quede descompensado)

Dividir antes de transformar: Para evitar la fuga de datos (data leakage); las transformaciones (como imputar nulos o escalar) deben aprenderse únicamente con los datos de entrenamiento (fit) y luego aplicarse al de test (transform), simulando datos nuevos reales

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [51]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
print(X_train.loc[df["edad"]>120,["edad"]])
X_train.loc[df["edad"]>120,["edad"]] = np.nan
print(X_train["edad"].isna().sum())

print(X_test.loc[df["edad"]>120,["edad"]])
X_test.loc[df["edad"]>120,["edad"]]=np.nan
print(X_test["edad"].isna().sum())

# TODO: contar ausentes en edad en entrenamiento y prueba

     edad
137   195
1
Empty DataFrame
Columns: [edad]
Index: []
0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [52]:
# TODO: imputer_numerico = SimpleImputer(...)
""""
strategy:
- Media: mean
- Mediana: median
- Moda: most frquent

"""


imputer_numerico = SimpleImputer(strategy="median")
# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| `edad` | Mediana (`median`) | Es numérica y tiene un outlier extremo (195 años) que distorsionaría la media. |
| `antiguedad_meses` | Mediana (`median`) | Es numérica y la mediana es más robusta frente a posibles valores extremos o asimetrías. |
| `ingresos_anuales` | Mediana (`median`) | Es numérica continua con valores ausentes y evita que ingresos muy altos sesguen el valor imputado. |
| `gasto_mensual` | Mediana (`median`) | Es numérica con asimetría positiva y valores atípicos altos, por lo que la mediana representa mejor el centro. |
| `dispositivo` | Moda (`most_frequent`) | Es categórica (texto) y no se puede calcular ni media ni mediana, así que se usa la categoría más frecuente. |
| `zona` | Moda (`most_frequent`) | Es categórica nominal y requiere imputarse con el valor más habitual del conjunto de entrenamiento. |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

**Respuesta:** Para evitar la fuga de datos (data leakage). Si ajustamos (`fit`) el imputador con todo el dataset, el conjunto de prueba influirá en el cálculo de las medianas y modas, contaminando la evaluación con información del futuro que el modelo no tendría en un entorno real.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [53]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(sparse_output=False)

# TODO: aplicar fit_transform a ejemplo
ejemplo_codificado = encoder_nominal.fit_transform(ejemplo)

# TODO: consultar get_feature_names_out()
print("Nombres de columnas:", encoder_nominal.get_feature_names_out())
print("\nMatriz codificada:\n", ejemplo_codificado)

Nombres de columnas: ['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']

Matriz codificada:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: 

- Columnas generadas: Se generan 3 columnas (dispositivo_Móvil, dispositivo_Ordenador y dispositivo_Tablet)

- Qué representan: Cada columna es una variable binaria que indica si la muestra pertenece a ese dispositivo (1) o no (0)

- Por qué no usar 0, 1 y 2!!!!
Porque dispositivo es una variable nominal sin orden natural; Si le asignams números consecutivos haría creer al modelo que existe una jerarquía o distancia matemática falsa (por ejemplo, que Tablet es el doble que Ordenador)

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [54]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1)
# TODO: transformar ejemplo_fidelidad y mostrar el resultado
resultado = encoder_ordinal.fit_transform(ejemplo_fidelidad)
print(resultado)
print(resultado.ravel())

[[2.]
 [0.]
 [3.]
 [1.]]
[2. 0. 3. 1.]


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: 

- Representan la posición jerárquica ordenada de cada nivel de fidelidad de menor a mayor: 0 = Bronce, 1 = Plata, 2 = Oro y 3 = Platino.

- No. Aunque los números estén separados por una unidad (0, 1, 2, 3), solo indican un orden relativo; el salto real en gasto o beneficios de pasar de Bronce a Plata no tiene por qué ser equivalente al de pasar de Oro a Platino.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [55]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
zscore = StandardScaler()
# TODO: transformar datos con cada uno
comparacion = {
    "Original": datos["valor"],
    "Min-Max": minmax.fit_transform(datos).ravel(),
    "StandardScaler": zscore.fit_transform(datos).ravel()
}
# TODO: mostrar tabla comparativa
miTabla = pd.DataFrame(comparacion).round(3)
miTabla

,Original,Min-Max,StandardScaler
0,10,0.00,-1.414
1,20,0.25,-0.707
2,30,0.50,0.000
3,40,0.75,0.707
4,50,1.00,1.414


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.00 | -1.414 |
| 20 | 0.25 | -0.707 |
| 30 | 0.50 | 0.000 |
| 40 | 0.75 | 0.707 |
| 50 | 1.00 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

**Respuesta:**
- **Rango de Min-Max:** De **0.00 a 1.00**.
- **Media tras estandarizar (`StandardScaler`):** **0** (0.000).
- **¿Pueden aparecer valores negativos?:** Sí, en `StandardScaler` todos los valores originales que estén por debajo de la media pasan a ser negativos 
- **Escalado para algoritmos basados en distancias:** Elegiría **`StandardScaler`** (estandarización) si existen outliers (como en nuestro dataset), ya que `MinMaxScaler` comprime la mayoría de los datos en un intervalo muy estrecho cuando hay valores extremos; si los datos están limpios y acotados sin outliers, `MinMaxScaler` también funciona bien al dejar todas las variables exactamente entre 0 y 1

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [56]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]


# TODO: comprobar que las columnas existen en X_train
todas = numericas + nominales + ordinales
faltantes = [col for col in todas if col not in X_train.columns]

print("Columnas faltantes:", faltantes)
print("¿Coincide el total de columnas?:", len(todas) == X_train.shape[1])

Columnas faltantes: []
¿Coincide el total de columnas?: True


 **Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: 
- ingresos_anuales (que se mueve en decenas de miles, alrededor de 36.000) frente a variables de escala muy pequeña como satisfaccion (1 a 5) o devoluciones

- No. Es fundamental para modelos basados en distancias o gradientes (como KNN, SVM, regresión logística o redes neuronales), pero no es necesario para modelos basados en árboles de decisión (como Random Forest o XGBoost), que hacen cortes por umbrales y son invariantes a la escala

- Aunque conceptualmente es una variable categórica ordinal, en este pipeline se trata como numérica asumiendo una distancia uniforme entre sus valores

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [57]:
# TODO: pipeline_numerico = Pipeline(steps=[
#     ("imputer", ...),
#     ("scaler", ...)
# ])

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Aseguramos la existencia de imputer_numerico por si no se ejecutó antes
if 'imputer_numerico' not in globals():
    imputer_numerico = SimpleImputer(strategy="median")

scaler_numerico = StandardScaler()

pipeline_numerico = Pipeline(steps=[
     ("imputer", imputer_numerico),
     ("scaler", scaler_numerico)
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Porque el escalador necesita calcular estadísticos como la media y la desviación estándar, y no puede operar matemáticamente con valores ausentes (NaN). Además, la mayoría de los transformadores y modelos de scikit-learn lanzan un error si reciben datos vacíos.

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [58]:
# TODO: pipeline_nominal = Pipeline(steps=[
#     ("imputer", ...),
#     ("encoder", ...)
# ])

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Aseguramos las dependencias locales
if 'imputer_categorico' not in globals():
    imputer_categorico = SimpleImputer(strategy="most_frequent")
if 'encoder_nominal' not in globals():
    encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

pipeline_nominal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", encoder_nominal)
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [59]:
# TODO: pipeline_ordinal = Pipeline(steps=[
#     ("imputer", ...),
#     ("encoder", OrdinalEncoder(
#         categories=[[...]],
#         handle_unknown="use_encoded_value",
#         unknown_value=-1
#     ))
# ])


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Aseguramos el imputador
if 'imputer_categorico' not in globals():
    imputer_categorico = SimpleImputer(strategy="most_frequent")

pipeline_ordinal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [64]:
# TODO: preprocesador = ColumnTransformer(transformers=[
#     ("num", pipeline_numerico, numericas),
#     ("nom", pipeline_nominal, nominales),
#     ("ord", pipeline_ordinal, ordinales)
# ])

from sklearn.compose import ColumnTransformer

# Definimos las variables por si acaso no se ejecutó la celda 12 en el kernel
numericas = ["edad", "antiguedad_meses", "ingresos_anuales", "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"]
nominal = ["dispositivo", "zona"]
ordinal = ["nivel_fidelidad"]

preprocesador = ColumnTransformer(transformers=[
     ("num", pipeline_numerico, numericas),
     ("nom", pipeline_nominal, nominal),
     ("ord", pipeline_ordinal, ordinal)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [61]:
# TODO: X_train_preparado = preprocesador.fit_transform(...)
# TODO: X_test_preparado = preprocesador.transform(...)

import numpy as np
import pandas as pd

# En caso de que no se hayan ejecutado las celdas de carga y split en tu sesión actual
if 'X_train' not in globals():
    df = pd.read_csv("clientes_ecommerce.csv")
    X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
    y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
         X, y, test_size=0.2, random_state=42, stratify=y
    )
    # Aplicar la regla didáctica de validación del Ejercicio 7
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train.loc[X_train["edad"] > 120, "edad"] = np.nan
    X_test.loc[X_test["edad"] > 120, "edad"] = np.nan

X_train_preparado = preprocesador.fit_transform(X_train)
X_test_preparado = preprocesador.transform(X_test)

**Responde:** 

- ¿Por qué no usar fit_transform() en prueba?: Para evitar data leakage (fuga de datos); los parámetros deben calcularse solo con entrenamiento y aplicarse a prueba con transform().

- ¿Qué aprende fit()?: Las medianas y modas para imputar, las medias y desviaciones típicas para escalar, y las categorías únicas de cada variable categórica.

- ¿Qué ocurre si aparece una categoría nominal nueva en prueba?: Si OneHotEncoder tiene handle_unknown="ignore", pone todo a 0 sin fallar; si está por defecto ("error"), lanza una excepción y detiene el código.

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [66]:
# TODO: mostrar X_train_preparado.shape y X_test_preparado.shape
if "X_train_preparado" not in globals():
    X_train_preparado = preprocesador.fit_transform(X_train)
    X_test_preparado = preprocesador.transform(X_test)
print("Shape test preparado:", X_test_preparado.shape)

# TODO: contar np.isnan(...) en ambas matrices
print("NaNs en train:", np.isnan(X_train_preparado).sum())
print("NaNs en test:", np.isnan(X_test_preparado).sum())

# TODO: consultar dtype y get_feature_names_out()
print("Dtype train:", X_train_preparado.dtype)
print("Columnas generadas:", preprocesador.get_feature_names_out())

# TODO: verificar filas de matrices frente a len(y_train), len(y_test)
print("Filas train == len(y_train):", X_train_preparado.shape[0] == len(y_train))
print("Filas test == len(y_test):", X_test_preparado.shape[0] == len(y_test))

# TODO: verificar que entrenamiento y prueba tienen igual número de columnas
print("Mismo número de columnas:", X_train_preparado.shape[1] == X_test_preparado.shape[1])

Shape test preparado: (80, 16)
NaNs en train: 0
NaNs en test: 0
Dtype train: float64
Columnas generadas: ['num__edad' 'num__antiguedad_meses' 'num__ingresos_anuales'
 'num__gasto_mensual' 'num__num_compras' 'num__devoluciones'
 'num__satisfaccion' 'nom__dispositivo_Móvil' 'nom__dispositivo_Ordenador'
 'nom__dispositivo_Tablet' 'nom__zona_Centro' 'nom__zona_Este'
 'nom__zona_Norte' 'nom__zona_Oeste' 'nom__zona_Sur'
 'ord__nivel_fidelidad']
Filas train == len(y_train): True
Filas test == len(y_test): True
Mismo número de columnas: True


**Responde:** 

- ¿Por qué aumentaron las columnas?: Pasaron de 10 a 16 por el OneHotEncoder, que desdobla las variables nominales en una columna binaria por cada categoría única (dispositivo generó 3 columnas y zona generó 5).

- ¿Coinciden las características en ambos conjuntos?: Sí, ambos conjuntos tienen exactamente las mismas 16 columnas en el mismo orden porque X_test se transformó usando el preprocesador ya ajustado en X_train.

- ¿Quedan ausentes?: No (0 valores NaN en ambos conjuntos), ya que todos los nulos fueron imputados.

- ¿Están alineadas las etiquetas?: Sí, el número de filas de X_train_preparado (320) y X_test_preparado (80) coincide exactamente con y_train e y_test, manteniendo intacto el orden de los registros.

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [67]:
# TODO: columnas_finales = preprocesador.get_feature_names_out()
columnas_finales = preprocesador.get_feature_names_out()

# TODO: df_train_preparado = pd.DataFrame(..., columns=..., index=X_train.index)
df_train_preparado = pd.DataFrame(X_train_preparado, columns=columnas_finales, index=X_train.index)

# TODO: df_test_preparado = pd.DataFrame(..., columns=..., index=X_test.index)
df_test_preparado = pd.DataFrame(X_test_preparado, columns=columnas_finales, index=X_test.index)

# TODO: mostrar head() de ambos
display(df_train_preparado.head())
display(df_test_preparado.head())

,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
360,-0.068723,-1.095565,0.651045,-1.086877,0.277945,-0.185205,-0.254293,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
356,-0.677904,1.129874,1.422884,-1.092939,0.277945,-1.069767,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
138,0.845048,-1.540653,0.907575,0.537957,-0.457116,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,-1.408922,0.447406,0.476223,0.303390,1.380537,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
216,-0.495150,1.456272,-0.493983,1.466735,0.645476,-1.069767,1.495680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.053113,-1.540653,-0.994595,-0.887364,1.013006,1.583919,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,-0.495150,1.040856,-0.185324,-0.504414,-0.089586,-1.069767,-0.254293,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,1.697902,-1.036220,0.611217,-0.253507,-0.457116,1.583919,-0.254293,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.784130,-1.095565,0.373354,-0.176812,-1.559708,1.583919,0.620694,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,-0.312396,0.269371,-1.092628,1.067445,1.013006,-1.069767,1.495680,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
